# Bagging : exercice sur le sous-échantillonnage de features

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/moncoachdata/cercle-des-experts/blob/main/ml-classique-et-deep-learning/bagging/bagging-exercice.ipynb)

📖 **Le mini-cours complet** : [Comprendre enfin pourquoi le bagging marche (et pas seulement comment)](https://www.monshotdata.com/p/bagging)

🧭 **Le cours** : [ML classique et Deep Learning › 2.1 Fondations mathématiques](https://www.monshotdata.com/mini-cours/ml-classique-et-deep-learning#h2sc21)

---

Tu as vu pourquoi décorréler les arbres rend la forêt meilleure. À toi d'implémenter la seconde source de hasard d'une Random Forest : **chaque arbre ne voit qu'une partie des features**. Tu complètes quelques lignes de code, puis tu compares ta forêt à un arbre seul et à la Random Forest de scikit-learn.

**À la fin de ce notebook, tu sauras :**

- tirer un sous-ensemble de colonnes, puis dupliquer des colonnes pour garder la même taille de dataset ;
- faire de même avec les lignes (bootstrap classique) ;
- mesurer ce que le sous-échantillonnage de features apporte, ou pas, par rapport à un arbre seul et à une Random Forest.


### Avant de lancer

**Comment faire l'exercice :** complète les lignes marquées `À COMPLÉTER` dans la fonction `sampling_function`, entre « TON CODE COMMENCE ICI » et « TON CODE SE TERMINE ICI », puis exécute le notebook de haut en bas. Une solution de référence est fournie dans `bagging-solution.ipynb`, à ouvrir après avoir essayé.

⚠️ **Conseil du mini-cours :** le jeu de données ci-dessous est en 2D, ce qui limite l'intérêt du sous-échantillonnage de features. Une fois l'exercice réussi, remplace-le par un dataset de grande dimension (par exemple `n_features=10` dans `make_classification`) : c'est là que le sous-échantillonnage de features a du sens.

🧪 *Testé le 20 novembre 2026 avec Python 3.13, NumPy, pandas, scikit-learn et Matplotlib dans leurs versions de Colab.*

## 1. Mise en place

In [1]:
import seaborn as sns

sns.set()
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression

import warnings
warnings.filterwarnings("ignore")

In [2]:
import logging
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)  # évite les avertissements si la police n'est pas installée

plt.rcParams['font.family'] = ['Comic Sans MS', 'sans-serif']

colors = ['#fe7c73', '#2471A3']

## 2. Les deux sources de hasard d'une Random Forest

Une Random Forest décorrèle ses arbres de deux façons :

1. **Sous-échantillonnage des lignes** (bootstrap) : chaque arbre est entraîné sur un échantillon différent des données ;
2. **Sous-échantillonnage des features** : chaque arbre ne voit qu'une partie des colonnes.

L'exercice porte sur la seconde idée, en conservant la même taille de dataset grâce à des **duplications**.

### 2.1 La fonction de bootstrapping

Tu dois écrire ta solution entre « TON CODE COMMENCE ICI » et « TON CODE SE TERMINE ICI » dans la fonction `sampling_function`, en deux endroits : l'échantillonnage des **colonnes**, puis celui des **lignes**. Dans les deux cas : on tire d'abord un échantillon initial, puis on le complète par des duplications aléatoires jusqu'à retrouver la taille d'origine.

**Convention à respecter :** nomme chaque colonne dupliquée `<nom d'origine>__dup<j>` (par exemple `X1__dup0`). Les cellules d'entraînement et de prédiction plus bas s'en servent pour retrouver quelles colonnes chaque arbre a vues.

In [ ]:
def sampling_function(df, row_sample_ratio, feature_sample_ratio):

    """
    Cette fonction accepte les arguments suivants :

    1. dataframe df de taille (n,d), où :
    -- n : nombre d'échantillons d'entraînement dans l'ensemble d'entraînement complet
    -- d : nombre de dimensions dans l'ensemble d'entraînement complet

    2. row_sample_ratio : La fraction de lignes à échantillonner avant duplication lors du bootstrapping des échantillons.

    3. feature_sample_ratio : La fraction de colonnes à échantillonner avant duplication lors du bootstrapping des colonnes.

    Cette fonction retourne un autre dataframe de taille (n,d) qui contient :
    - quelques lignes dupliquées
    - quelques colonnes dupliquées

    """

    # définir le nombre total d'échantillons et de colonnes
    total_samples, total_columns = df.shape
    final_size = total_samples  # taille finale du dataset rééchantillonné

    # ÉCHANTILLONNAGE DES COLONNES :

    initial_column_size = int(feature_sample_ratio*total_columns)

    ### TON CODE COMMENCE ICI ###

    # échantillonner aléatoirement 'initial_column_size' colonnes de 'df' SAUF LA COLONNE Y
    initial_column_sample_df = ...  # À COMPLÉTER

    remaining_columns = total_columns - initial_column_size

    new_columns = pd.DataFrame()
    for j in range(remaining_columns):

        # échantillonner aléatoirement une colonne de 'initial_column_sample_df'
        sampled_column = ...  # À COMPLÉTER

        # l'ajouter au df 'new_columns' avec son nom de colonne
        new_columns[...] = ...  # À COMPLÉTER : nom f"{nom_d_origine}__dup{j}", puis la colonne

    ### TON CODE SE TERMINE ICI ###

    # on remet la colonne cible y
    column_sampled_df = pd.concat((initial_column_sample_df, new_columns, df[['y']]), axis = 1)

    # ÉCHANTILLONNAGE DES LIGNES

    # nombre de lignes à échantillonner
    initial_sample_size = int(row_sample_ratio*total_samples)

    ### TON CODE COMMENCE ICI ###

    # échantillonner aléatoirement 'initial_sample_size' lignes de 'column_sampled_df'
    initial_row_sample_df = ...  # À COMPLÉTER

    new_rows = pd.DataFrame()

    remaining_rows = final_size - initial_sample_size
    for i in range(remaining_rows):

        # échantillonner aléatoirement une ligne de 'column_sampled_df'
        sampled_row = ...  # À COMPLÉTER

        # l'ajouter au df 'new_rows'
        new_rows = pd.concat((new_rows, sampled_row))

    bootstrapped_dataset = pd.concat((initial_row_sample_df, new_rows))
    return bootstrapped_dataset

### 2.2 La fonction de prédiction

Chaque arbre prédit, puis on retient la classe la plus fréquente (vote majoritaire). Rien à modifier ici.

In [ ]:
from collections import Counter

def predict(models, test_data):
    # En supposant que vous avez des données de test 'X_test' sous forme de DataFrame
    predictions = []

    # Faire des prédictions en utilisant chaque arbre dans 'all_trees'
    for model in models:
        # chaque arbre ne regarde que les colonnes (d'origine) qu'il a vues à l'entraînement
        model_predictions = model.predict(test_data[model.cols_].values)
        predictions.append(model_predictions)

    # Effectuer un vote majoritaire pour agréger les prédictions
    ensemble_predictions = []
    for i in range(len(test_data)):
        ensemble_prediction = Counter([prediction[i] for prediction in predictions]).most_common(1)[0][0]
        ensemble_predictions.append(ensemble_prediction)

    ensemble_predictions = np.array(ensemble_predictions)

    return ensemble_predictions

## 3. Les données et les modèles de référence

Un jeu de classification en 2D, séparé en entraînement (1000 points) et test (200 points). Pense à passer à plus de features une fois l'exercice réussi.

In [ ]:
# Générer des données synthétiques pour entraîner l'arbre de décision
X, y = make_classification(
    n_samples=1200,
    n_features=2,
    n_clusters_per_class=1,
    n_informative=2, class_sep=0.5,
    n_redundant=0,
    n_repeated=0,
    random_state=21
)

fig, ax = plt.subplots()
plt.scatter(X[:, 0], X[:, 1], c=[colors[i] for i in y])
ax.set_xticklabels([])
ax.set_yticklabels([])
plt.show()

X_train, y_train = X[:1000], y[:1000]
X_test, y_test = X[1000:], y[1000:]

### 3.1 Un arbre de décision seul

In [ ]:
dtree = DecisionTreeClassifier(random_state=0)
dtree.fit(X_train, y_train)

xx, yy = np.meshgrid(np.linspace(np.floor(X_train[:, 0].min()), np.ceil(X_train[:, 0].max()), 100), np.linspace(np.floor(X_train[:, 1].min()), np.ceil(X_train[:, 1].max()), 100))
Z = dtree.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

fig, ax = plt.subplots()
plt.contourf(xx, yy, Z, alpha=0.5, cmap='coolwarm_r')
ax.set_title("Arbre de décision", fontsize = 20, fontweight = "bold", pad=15)
ax.set_xticklabels([])
ax.set_yticklabels([])
plt.show()

dtree.score(X_test, y_test)

### 3.2 La Random Forest de scikit-learn

In [ ]:
rfmodel = RandomForestClassifier(max_features="sqrt", max_samples=0.5, n_estimators=21)
rfmodel.fit(X_train, y_train)

xx, yy = np.meshgrid(np.linspace(np.floor(X_train[:, 0].min()), np.ceil(X_train[:, 0].max()), 100), np.linspace(np.floor(X_train[:, 1].min()), np.ceil(X_train[:, 1].max()), 100))
Z = rfmodel.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

fig, ax = plt.subplots()
plt.contourf(xx, yy, Z, alpha=0.5, cmap='coolwarm_r')
ax.set_title("Random Forest", fontsize = 20, fontweight = "bold", pad=15)
ax.set_xticklabels([])
ax.set_yticklabels([])
plt.show()

rfmodel.score(X_test, y_test)

## 4. Ta Random Forest

### 4.1 Les données en DataFrame

In [ ]:
df_train = pd.DataFrame()
df_test = pd.DataFrame()

df_train[["X1", "X2"]] = X_train
df_train["y"] = y_train

df_test[["X1", "X2"]] = X_test
df_test["y"] = y_test

### 4.2 Entraîner les arbres

Chaque arbre est entraîné sur un dataset rééchantillonné par ta fonction. Cette cellule ne fonctionnera qu'une fois `sampling_function` complétée.

In [ ]:
total_models = 21
row_sample_ratio, feature_sample_ratio = 0.5, 0.5

all_trees = []

for i in range(total_models):
    data = sampling_function(df_train, row_sample_ratio, feature_sample_ratio)

    features = [c for c in data.columns if c != 'y']
    model = DecisionTreeClassifier(max_features="sqrt")
    model.fit(data[features].values, data['y'])
    model.cols_ = [c.split('__dup')[0] for c in features]   # colonnes d'origine vues par cet arbre

    all_trees.append(model)

### 4.3 Agréger les prédictions et comparer

On trace la frontière de décision de ta forêt et on compare sa précision sur le test à celle des deux modèles de référence.

In [ ]:
xx, yy = np.meshgrid(np.linspace(np.floor(X_train[:, 0].min()), np.ceil(X_train[:, 0].max()), 100), np.linspace(np.floor(X_train[:, 1].min()), np.ceil(X_train[:, 1].max()), 100))

grid = pd.DataFrame(
    np.c_[xx.ravel(), yy.ravel()],
    columns=["X1", "X2"]
)

Z = predict(all_trees, grid)
Z = Z.reshape(xx.shape)

fig, ax = plt.subplots()
plt.contourf(xx, yy, Z, alpha=0.5, cmap='coolwarm_r')
ax.set_title("Random Forest Personnalisée", fontsize = 20, fontweight = "bold", pad = 15)
ax.set_xticklabels([])
ax.set_yticklabels([])
plt.show()

sum(predict(all_trees, df_test[["X1", "X2"]]) == df_test.y)/df_test.shape[0]

In [ ]:
dtree_accuracy = dtree.score(X_test, y_test)
rf_accuracy = rfmodel.score(X_test, y_test)
custom_rf_accuracy = sum(predict(all_trees, df_test[["X1", "X2"]]) == df_test.y)/df_test.shape[0]

print(f"Précision de l'arbre de décision sur le test : {dtree_accuracy}")
print(f"Précision de la Random Forest sur le test : {rf_accuracy}")
print(f"Précision de la Random Forest personnalisée sur le test : {custom_rf_accuracy}")

---

## Pour aller plus loin

- [Comprendre enfin pourquoi le bagging marche (et pas seulement comment)](https://www.monshotdata.com/p/bagging) — le mini-cours complet et le notebook de démonstration